### Import Libraries

In [ ]:
#import flwr libraries
import flwr as fl
from flwr.common import (
    ndarrays_to_parameters,
)
from flwr.client import Client, ClientApp
from flwr.simulation import run_simulation
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import Strategy
from flwr.simulation.ray_transport.utils import enable_tf_gpu_growth

from typing import Dict, List

import tensorflow as tf
import numpy as np
import os
import random
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.rc('font', size=16)

os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

from resnet_model import build_ResNet
from FlowerClient import *
from GetClient import *
from ClientConfigurationStrategies import *

tfk = tf.keras
tfkl = tf.keras.layers
print(tf.__version__)

2025-03-05 17:12:44,805	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2.18.0


In [2]:
# Random seed for reproducibility
seed = 42

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
tf.compat.v1.set_random_seed(seed)

### Load Dataset

In [3]:
train_data = np.loadtxt(f"./Datasets/Datasets_Training/StarLightsCurves/StarLightCurves_TRAIN.txt")
test_data = np.loadtxt(f"./Datasets/Datasets_Training/StarLightsCurves/StarLightCurves_TEST.txt")

#np.random.shuffle(train_data)
#Swap x_train and x_test because the test set is bigger
X_train = np.expand_dims(train_data[:, 1:],axis=-1)
y_train = train_data[:, 0]

X_test = np.expand_dims(test_data[:, 1:], axis=-1)
y_test = test_data[:, 0]

#Take the 10% as validation set
validation_sample = (int)(0.1 * X_train.shape[0])

#Take the last 1000 samples of x_train for the validation step
X_val = X_train[-validation_sample:]
y_val = y_train[-validation_sample:]

X_train = X_train[:-validation_sample]
y_train = y_train[:-validation_sample]

X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((900, 1024, 1), (900,), (100, 1024, 1), (100,), (8236, 1024, 1), (8236,))

### Dataset Split and Graphical Distribution

In [ ]:
def splitting_dataset(dataset: np.ndarray, num_clients: int, file_path: str = None, from_configuration: bool = False):
  total_points = dataset.shape[0]
  indexes = []
  #Initialize with the first index
  indexes.append(0)

  #Retrieve splitting from configuration according to the data_volume parameter defined
  if from_configuration:
    with open(file_path, "r") as json_file:
      data = json.load(json_file)
    for obj_exp in data:
      number_points = (int)(obj_exp['data_volume']*total_points)
      new_index = indexes[-1] + number_points
      indexes.append(new_index)
    
  else:
    #Uniform splitting
    single_set = int(total_points/num_clients)
    for client in range(0,num_clients):
      new_index = indexes[-1] + single_set
      indexes.append(new_index)

  return indexes

#Dataset Split and Graphica distribution
column_name = ["Classes"]
clients = 10
indexes_train = splitting_dataset(dataset=X_train, num_clients=clients)
print(indexes_train)
indexes_val = splitting_dataset(dataset=X_val, num_clients=clients)
print(indexes_val)


for i in range(0,clients):
  start_index = indexes_train[i]
  end_index = indexes_train[i + 1]
  df = pd.DataFrame(y_train[start_index:end_index])
  df.columns = column_name

  # Inspect activities timestamps
  plt.figure(figsize=(6,4))
  sns.countplot(x = 'Classes', data = df, order = df.Classes.value_counts().index)
  plt.title(f'Client_{i}')
  plt.show()

for i in range(0,clients):
  start_index = indexes_val[i]
  end_index = indexes_val[i + 1]
  df = pd.DataFrame(y_val[start_index:end_index])
  df.columns = column_name

  # Inspect activities timestamps
  plt.figure(figsize=(6,4))
  sns.countplot(x = 'Classes', data = df, order = df.Classes.value_counts().index)
  plt.title(f'Client_{i}')
  plt.show()

  df = pd.DataFrame(y_test)
  df.columns = column_name

# Inspect activities timestamps
plt.figure(figsize=(6,4))
sns.countplot(x = 'Classes', data = df, order = df.Classes.value_counts().index)
plt.title(f'Test')
plt.show()

### Load Parameters and Dataset split

In [5]:
#This is done to poison the data points correctly
Y_TRAIN = y_train

# Convert the sparse labels to categorical values
y_train = tfk.utils.to_categorical(y_train)
y_val = tfk.utils.to_categorical(y_val)
y_test = tfk.utils.to_categorical(y_test)

X_TRAIN, X_VAL, Y_VAL, X_TEST, Y_TEST = X_train, X_val, y_val, X_test, y_test

input_shape = X_train.shape[1:]
classes = y_train.shape[-1]

In [6]:
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

SystemError: GPU device not found

### Simulation

In [7]:
NUM_CLIENTS = 10
BATCH_SIZE = 64
FRACTION_FIT = 0.2
NUM_SET_CLIENTS = 10 #Only for a vertical configuration where only a subset of all the clients are considered available for training
DATA_VOLUME = 1.0
CONSISTENCY = 0.0
ACCURACY = 0.0
COMPLETENESS = 0.0
POISONING_TYPE = 'volume' 
SAVE_EXP_RESULTS = True

# Save Results
file_path = f"./Datasets/Datasets_Training/StarLightsCurves/results/new_results.json"
if not os.path.exists(file_path):
  options = [{'experiment_method': 'vertical_volume', 'data_volume': DATA_VOLUME, 'num_clients': (int)(NUM_CLIENTS*FRACTION_FIT), 'effective_epochs': 0, 'effective_emissions_kg': 0, 'effective_energy_consumed': 0, 'effective_duration': 0}]
  with open(file_path, "w") as json_file:
    json.dump(options, json_file)
else:
  options = {'experiment_method': 'vertical_volume', 'data_volume': DATA_VOLUME, 'num_clients': (int)(NUM_CLIENTS*FRACTION_FIT), 'effective_epochs': 0, 'effective_emissions_kg': 0, 'effective_energy_consumed': 0, 'effective_duration': 0}
  with open(file_path, "r") as json_file:
    data = json.load(json_file)
    data.append(options)
  with open(file_path, "w") as json_file:
    json.dump(data, json_file)


new_model = build_ResNet(input_shape, classes)

In [8]:
# Function to initialize flower node client with its data partition (x_train_p refers to the all x_train set from which the partition is computed)
# Per simulation only data quality dimension (together with or without data volume) can be poisoned 
def get_client_fn_simulation(volume, quality, poisoning_type, x_train_p, y_train_p, x_val_p, y_val_p, indexes_train, indexes_val):
  # It must be called to create an instance of a new FlowerClient
  def client_fn(context: Context) -> FlowerClient:
      """Create a Flower client representing a single organization."""

      # Note: each client gets a different trainloader/valloader, so each client will train and evaluate on their own unique data
      print("Client with CID: {}\n".format((int)(context.node_config["partition-id"])))
      options_volume = {'data_quality_dimension_percentage': volume, 'experiment_method': 'uniform'} # uniform over the classes
      options_quality = {'data_quality_dimension_percentage': quality, 'experiment_method': 'uniform'} # uniform over the classes
      (x_train, y_train), (x_test, y_test) = load_partition((int)(context.node_config["partition-id"]), x_train_p, y_train_p, x_val_p, y_val_p, indexes_train, indexes_val)

      # Data quality poisoning
      x_train, y_train = reduce_data_volume(x_train, y_train, options_volume) #Data Volume can be poisoned alone or together other data quality dimensions
      # Reduce Consistency Horizontally
      if poisoning_type == 'consistency':
        x_train, y_train = reduce_consistency(x_train, y_train, options_quality)
      # Reduce Accuracy Horizontally
      if poisoning_type == 'accuracy':
        x_train, y_train = reduce_accuracy(x_train, y_train, options_quality)
      # Reduce Accuracy Horizontally
      if poisoning_type == 'completeness':
        x_train, y_train = reduce_completeness(x_train, y_train, options_quality)
      # This is just to take the right shape and build the model
      y_train = tfk.utils.to_categorical(y_train)

      # Load model
      model_client = build_ResNet(x_train.shape[1:], y_train.shape[-1])

      # Create a  single Flower client representing a single organization
      return FlowerClient(context.node_config["partition-id"], model_client, x_train, y_train, x_test, y_test).to_client()
  return client_fn

In [9]:
# Create strategy | add which strategies you want to test from ClientConfigurationStrategies
strategy_basic = FedCustom(
    file_path = file_path,
    fraction_fit=FRACTION_FIT,
    fraction_evaluate=0.1,
    min_fit_clients=1,
    min_evaluate_clients=1,
    min_available_clients=NUM_CLIENTS,
    #evaluate_fn=get_evaluate_fn(new_model, X_test, y_test, file_path),
    # on_fit_config_fn=fit_config,
    #on_evaluate_config_fn=evaluate_config,
    initial_parameters=ndarrays_to_parameters(new_model.get_weights()),
)

def server_fn(context: Context) -> ServerAppComponents:
    # Configure the server for just 3 rounds of training
    config = ServerConfig(num_rounds=3)
    return ServerAppComponents(
        config=config,
        strategy=strategy_basic,  # <-- pass the new strategy here
    )

# Create the ServerApp
server = ServerApp(server_fn=server_fn)

In [ ]:
#Parameters for vertical and mixed reduction configurations (nodes and data volume) 
'''
OFFSET = 0.1
num_clients = int(NUM_CLIENTS * FRACTION_FIT)
number_clients_considered = (int)(NUM_CLIENTS * FRACTION_FIT)
volume_data = num_clients * 0.1 #0.125 is the base fraction of dataset per each client
#If we consider Vertical ClientConfigurationStrategy 
vertical_quality = (float)(OFFSET * (number_clients_considered - (int)(ACCURACY * (number_clients_considered)))) + ACCURACY
'''
# Run simulation new library
backend_config = {"client_resources": {"num_cpus": 1}}

client = ClientApp(client_fn=get_client_fn_simulation(DATA_VOLUME, ACCURACY, POISONING_TYPE, X_TRAIN, Y_TRAIN, X_VAL, Y_VAL, indexes_train, indexes_val))

run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=NUM_CLIENTS,
    backend_config=backend_config,
)